# Tutorial 5: Train NicheTrans on STARmap PLUS data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

/home/ubuntu/miniconda3/envs/NicheTrans/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialize the args and fix seeds

In [2]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.25, n_top_genes=2000, workers=4, AD_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/AD_model_adata_protein', Wild_type_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/wild_type_adata_protein', max_epoch=20, stepsize=20, train_batch=128, test_batch=32, optimizer='adam', lr=0.0001, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0', context_model_type='transformer', gnn_num_layers=2, gnn_hidden_dim=None, gnn_graph_type='full')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

------Calculating spatial graph...
The graph contains 124464 edges, 10372 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 115608 edges, 9634 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 96408 edges, 8034 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  10372 spots, 894.0 positive tao, 291.0 positive plaque 
  test     |   9634 spots, 620.0 positive tao, 195.0 positive plaque 
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader, device=device)
os.makedirs('./trained_model', exist_ok=True)
torch.save(model.state_dict(), './trained_model/NicheTrans_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 81/81	 Loss 0.479727 (0.546357)
==> Epoch 2/20
Batch 81/81	 Loss 0.368830 (0.419334)
==> Epoch 3/20
Batch 81/81	 Loss 0.271349 (0.341468)
==> Epoch 4/20
Batch 81/81	 Loss 0.242064 (0.291312)
==> Epoch 5/20
Batch 81/81	 Loss 0.214011 (0.258157)
==> Epoch 6/20
Batch 81/81	 Loss 0.230748 (0.234343)
==> Epoch 7/20
Batch 81/81	 Loss 0.193658 (0.218489)
==> Epoch 8/20
Batch 81/81	 Loss 0.184056 (0.204484)
==> Epoch 9/20
Batch 81/81	 Loss 0.185091 (0.195967)
==> Epoch 10/20
Batch 81/81	 Loss 0.167925 (0.187158)
==> Epoch 11/20
Batch 81/81	 Loss 0.146548 (0.180503)
==> Epoch 12/20
Batch 81/81	 Loss 0.145439 (0.175193)
==> Epoch 13/20
Batch 81/81	 Loss 0.160075 (0.170587)
==> Epoch 14/20
Batch 81/81	 Loss 0.170961 (0.166170)
==> Epoch 15/20
Batch 81/81	 Loss 0.129759 (0.161461)
==> Epoch 16/20
Batch 81/81	 Loss 0.176413 (0.157827)
==> Epoch 17/20
Batch 81/81	 Loss 0.146966 (0.154048)
==> Epoch 18/20
Batch 81/81	 Loss 0.149053 (0.149531)
==> Epoch 19/20
Batch 81/81	 Loss 0.1